In [ ]:
from biosteam import main_flowsheet as F
import biosteam as bst
import thermosteam as tmo
import pandas as pd
import numpy as np


from lignin_saf.ligsaf_chemicals import create_chemicals
from lignin_saf.ligsaf_settings import feed_parameters, prices
from lignin_saf.settings.process_params import additional_hdo
from lignin_saf.systems.rcf import create_rcf_system
from lignin_saf.systems.rcf_oil_purification import create_rcf_oil_purification_system
from lignin_saf.systems.monomer_purification import create_monomer_purification_system
from lignin_saf.systems.hdo import create_hdo_system
from lignin_saf.cellulosic_tea import create_cellulosic_ethanol_tea
from lignin_saf.ligsaf_units import HydrogenStorageTank



chems = create_chemicals()
bst.settings.set_thermo(chems)
bst.settings.CEPCI = 840

# Poplar group must be defined before creating any stream that references it
chems.define_group(
    name='Poplar',
    IDs=['Glucan', 'Xylan', 'Arabinan', 'Mannan', 'Galactan',
         'Sucrose', 'Lignin', 'Acetate', 'Extract', 'Ash'],
    composition=[0.464, 0.134, 0.002, 0.037, 0.014,
                 0.001, 0.285, 0.035, 0.016, 0.012],
    wt=True
)

poplar_in = bst.Stream('Poplar_In',
                       Poplar=feed_parameters['flow'] * 1e3,
                       Water=feed_parameters['moisture'] * feed_parameters['flow'] * 1e3,
                       phase='l', units='kg/d', price=prices['Feedstock'])

# ── Area 200: RCF process ──────────────────────────────────────────────────
rcf_system = create_rcf_system(ins=poplar_in)
rcf_system.simulate()

# ── Area 300: Purification ─────────────────────────────────────────────────
rcf_oil_purification_sys = create_rcf_oil_purification_system(ins=F.RCF_CRUDE_OUT)
monomer_purification_sys = create_monomer_purification_system(ins=F.PURE_OIL_OUT)
rcf_oil_purification_sys.simulate()
monomer_purification_sys.simulate()

# ── Area 400: Hydrodeoxygenation ───────────────────────────────────────────
hdo_system = create_hdo_system(ins=F.MON_MONOMERS_OUT)
hdo_system.simulate()

h2_rcf = bst.Stream()
h2_rcf.copy_like(F.RCF_H2_IN)

h2_hdo = bst.Stream()
h2_hdo.copy_like(F.HDO_H2_IN)

# Shared H2 storage — sized from combined ETJ + HDO fresh H2 demand
h2_feed_mixer = bst.Mixer('H2_FEED_MIX', ins=(h2_rcf, h2_hdo))
shared_h2_storage = HydrogenStorageTank('H2_TK', ins=h2_feed_mixer.outs[0])


WWT = bst.create_conventional_wastewater_treatment_system('WWT', ins=(F.WW_10, F.WastePulp, F.RCF_WW_OUTS, F.WW_11, F.WW_12, F.HDO_WW, F.HDO_wash_water))

for unit in WWT.units:
    if hasattr(unit, 'strict_moisture_content'):
        unit.strict_moisture_content = False


BT = bst.facilities.BoilerTurbogenerator('BT', fuel_price=prices['CH4'])


gas_mixer= bst.Mixer('MIX_BT_gas', ins=(WWT.outs[0], F.RCF_PSAWASTE_OUTS, F.HDO_purge_gases))

BT.ins[0] = WWT.outs[1]   # Connecting sludge to BT solids feed
BT.ins[1] = gas_mixer.outs[0]   # Connecting biogas from WW treatment and PSA waste gases from RCF




rcf_pure_mon_hdo_system = bst.System(
    'RCF_HDO_Combined_System',
    path=(rcf_system, rcf_oil_purification_sys, monomer_purification_sys, hdo_system, WWT),
    facilities=[shared_h2_storage, gas_mixer,  BT],
)

rcf_pure_mon_hdo_system.simulate()


c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\thermosteam\equilibrium\bubble_point.py:128: RuntimeWarning: Hydrogen has no defined Dortmund groups; functional group interactions are ignored
  self.gamma = thermo.Gamma(chemicals)
c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\thermosteam\equilibrium\bubble_point.py:128: RuntimeWarning: Methane has no defined Dortmund groups; functional group interactions are ignored
  self.gamma = thermo.Gamma(chemicals)
c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\thermosteam\equilibrium\dew_point.py:129: RuntimeWarning: Methane has no defined Dortmund groups; functional group interactions are ignored
  self.gamma = thermo.Gamma(chemicals)
c:\Users\hwadg\anaconda3\envs\pyfuel\lib\site-packages\biosteam\units\_pump.py:224: RuntimeWarning: <Pump: RCF_PUMP1> no pump type available at current power (2.45e+03 hp), head (3.35e+03 ft), kinematic viscosity (6.09e-07 m2/s), and NPSH (0.745 ft); assuming centrigugal pump
  warn(f'{repr

In [2]:
integrated_tea = create_cellulosic_ethanol_tea(rcf_pure_mon_hdo_system)

In [3]:

integrated_tea.operating_days = 330
mjsp = round(((integrated_tea.solve_price(F.HDO_CYCLOALKANES_OUT)*F.HDO_CYCLOALKANES_OUT.rho)/264.172),2)

print(f'The MSP for SAF cycloalkanes is  {mjsp} USD/gal')

The MSP for SAF cycloalkanes is  52.13 USD/gal


In [ ]:
model = bst.Model(rcf_pure_mon_hdo_system)

In [ ]:
from chaospy import distributions as shape
param = model.parameter

In [ ]:

var_50 = 0.5 # 50% variation in parameters - set for a few
var_20 = 0.2 # 20% variation in other parameters

In [ ]:
# Distillation col 1 light key recovery at the top
dist = shape.Uniform(lower = 0.7, 
                     upper = 0.9999) 
@param(name = 'Light key recovery - column 1',
    element = 'HDO', 
    kind = 'coupled',
    units = 'wt%',
    baseline = additional_hdo['hdo_col_1_light_dist_recovery'], distribution = dist)
def set_light_key_recovery_column_1(i):
    additional_hdo['hdo_col_1_light_dist_recovery'] = i
    F.unit.HDO_COL1.Lr = i


# Distillation col 1 heavy key recovery at the bottom
dist = shape.Uniform(lower = 0.0001, 
                     upper = 0.3) 
@param(name = 'Heavy key recovery - column 1',
    element = 'RCF', 
    kind = 'coupled',
    units = 'wt%',
    baseline = additional_hdo['hdo_col_1_heavy_bottom_recovery'], distribution = dist)
def set_heavy_key_recovery_column_1(i):
    additional_hdo['rcf_col_1_heavy_bottom_recovery'] = i
    F.unit.HDO_COL1.Hr = i


# Distillation col 1 k value
dist = shape.Uniform(lower = 0.0001, 
                     upper = 10) 
@param(name = 'Ratio of reflux to minimum reflux-  column 1',
    element = 'RCF', 
    kind = 'coupled',
    units = '-',
    baseline = additional_rcf['rcf_col_1_k'], distribution = dist)
def set_k_column_1(i):
    additional_rcf['rcf_col_1_k'] = i
    F.unit.RCF_COL1.k = i


# Distillation col 2 heavy key recovery at the bottom
dist = shape.Uniform(lower = 0.7, 
                     upper = 0.9999) 
@param(name = 'Light key mole fraction at top - column 2',
    element = 'RCF', 
    kind = 'coupled',
    units = 'wt%',
    baseline = additional_rcf['rcf_col_2_mol_frac_light_top'], distribution = dist)
def set_light_key_mol_fraction_top(i):
    additional_rcf['rcf_col_2_mol_frac_light_top'] = i
    F.unit.RCF_COL2.y_top  = i

# Distillation col 2 light key mole fraction at the bottom
dist = shape.Uniform(lower = 0.0001, 
                     upper = 0.3) 
@param(name = 'Light key mole fraction at bottom - column 2',
    element = 'RCF', 
    kind = 'coupled',
    units = 'wt%',
    baseline = additional_rcf['rcf_col_2_mol_frac_light_bottom'], distribution = dist)
def set_light_key_mol_fraction_bottom(i):
    additional_rcf['rcf_col_2_mol_frac_light_bottom'] = i
    F.unit.RCF_COL2.x_bot   = i


# Distillation col 2 k value
dist = shape.Uniform(lower = 0.0001, 
                     upper = 10) 
@param(name = 'Ratio of reflux to minimum reflux-  column 2',
    element = 'RCF', 
    kind = 'coupled',
    units = '-',
    baseline = additional_rcf['rcf_col_1_k'], distribution = dist)
def set_k_column_2(i):
    additional_rcf['rcf_col_2_k'] = i
    F.unit.RCF_COL2.k = i

